# PEMDT-Cirrhosis — Hepatic Decompensation Digital Twin

**Probabilistic & Explainable Bayesian model · MIMIC-IV (hosp + icu)**
Adapted from the PEMDT v2 architecture. Latent states: **Compensated / Decompensated**.

## What this predicts
A composite **hepatic decompensation** phenotype in a cirrhosis cohort, defined as the
presence (during the index cirrhosis admission) of any of: **ascites · variceal
hemorrhage · hepatic encephalopathy · spontaneous bacterial peritonitis (SBP) ·
hepatorenal syndrome**. The model produces a per-window `decomp_probability`
trajectory from physiology and is validated against this label.

## Pipeline
| Cell | Purpose | Key outputs |
|---|---|---|
| Config | Paths, itemids, ICD/drug maps, itemid verifier | — |
| Step 0 | Cirrhosis cohort + composite decompensation label + demographics + timestamped event anchors | `cirrhosis_cohort.csv` |
| Step 1 | Vital-sign extraction (chartevents) | — |
| Step 2 | Liver lab-panel extraction (labevents) | — |
| Step 3 | Unified timeline + MELD-Na | `cirrhosis_timeline.csv` |
| Step 5 | Bayesian decompensation HMM | `cirrhosis_dashboard.csv` |
| Step 6 | Glass-box LLR explainability | — |
| Run | Ties steps together | `cirrhosis_patient_summary.csv`, `population_stats.json` |
| Validation | ROC/PR vs label, MELD-Na baseline, lead-time, calibration | `validation_output/` |

## Required MIMIC-IV files
`hosp/` → `diagnoses_icd`, `d_icd_diagnoses`, `patients`, `admissions`, `labevents`, `d_labitems`, `prescriptions`
`icu/`  → `chartevents`, `d_items`, `procedureevents`
(`.csv` or `.csv.gz` both work.)

## Design notes — read before interpreting results
- **No label leakage.** Decompensation-specific drugs/procedures (lactulose, rifaximin,
  terlipressin, octreotide, paracentesis) are used **only** as *timestamped anchors for
  lead-time analysis* — they are **not** model features. The HMM sees only physiology
  (labs + vitals), so it is not just detecting treatments.
- **ICD codes are admission-level, not timestamped** in MIMIC-IV. They define the cohort
  and the composite phenotype for the *index admission*; precise within-stay onset timing
  for lead-time comes from the timestamped drug/procedure markers, which exist for a
  subset of patients only.
- **MELD-Na** is computed as a standard *clinical baseline comparator* in validation, not
  as the label (per the chosen design).
- **SBP antibiotic markers (ceftriaxone/cefotaxime) are low-specificity** — used cautiously.
- **Verify itemids.** The config cell prints a check of curated lab itemids against your
  `d_labitems`. Confirm the `[OK]`/`[CHECK]` report before trusting feature extraction.
- This notebook is presented **for review**, not as a validated clinical tool. It has not
  been executed here (the MIMIC files are not mounted in this session).


## Config — paths, itemids, ICD & drug maps

In [1]:
# ============================================================
# PEMDT-Cirrhosis — Configuration & Constants
# ============================================================
import os, gc, re, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
warnings.filterwarnings("ignore")

# ---- Data location (auto-detect; edit if needed) ----------
CANDIDATE_ROOTS = [
    "/Users/sumiyasayeed/Documents/mimic4/mimiciv",
    "../data/mimiciv", "./data/mimiciv", "../../data/mimiciv",
]
MIMIC_ROOT = next((p for p in CANDIDATE_ROOTS
                   if os.path.exists(os.path.join(p, "hosp"))), CANDIDATE_ROOTS[0])
HOSP = os.path.join(MIMIC_ROOT, "hosp")
ICU  = os.path.join(MIMIC_ROOT, "icu")
OUT  = Path(".")                       # outputs next to this notebook
print(f"MIMIC_ROOT = {MIMIC_ROOT}")
print(f"  hosp exists: {os.path.exists(HOSP)} | icu exists: {os.path.exists(ICU)}")

WINDOW       = "60min"     # cirrhosis dynamics are slower than acute ICU vitals
CHUNK_SIZE   = 1_000_000
MAX_PATIENTS = None        # e.g. 500 for a fast test; None = full cohort

def _csv(dirpath, name):
    """Return path to <name>.csv or <name>.csv.gz, whichever exists."""
    for ext in (".csv", ".csv.gz"):
        p = os.path.join(dirpath, name + ext)
        if os.path.exists(p):
            return p
    return os.path.join(dirpath, name + ".csv")

def _read(path, **kw):
    return pd.read_csv(path, **kw)

# ============================================================
# ITEMID MAPS  (verify against the dictionaries in the cohort cell)
# ============================================================
CHART_ITEMIDS = {                                  # icu/chartevents -> d_items
    "HR_bpm":[220045], "SBP":[220179,220050,224167,227243],
    "DBP":[220180,220051,224643,227242], "MAP":[220052,220181,225312],
    "SpO2":[220277,646], "Resp_rate":[220210,224422],
    "Temp_C":[223762,226329], "GCS_total":[220739,223900,223901],
}
VITAL_RANGE = {
    "HR_bpm":(20,300),"SBP":(40,260),"DBP":(20,200),"MAP":(20,200),
    "SpO2":(50,100),"Resp_rate":(4,60),"Temp_C":(30,43),"GCS_total":(3,15),
}

LAB_ITEMIDS = {                                    # hosp/labevents -> d_labitems
    "Bilirubin":[50885], "Albumin":[50862], "INR":[51237], "PT":[51274],
    "Platelets":[51265], "AST":[50878], "ALT":[50861], "AlkPhos":[50863],
    "Ammonia":[50866], "Creatinine":[50912], "Sodium":[50983],
    "WBC":[51301,51300], "Hemoglobin":[51222], "Lactate":[50813],
}
LAB_RANGE = {
    "Bilirubin":(0.1,60),"Albumin":(0.5,6),"INR":(0.5,20),"PT":(8,150),
    "Platelets":(5,1000),"AST":(1,20000),"ALT":(1,20000),"AlkPhos":(10,3000),
    "Ammonia":(5,1000),"Creatinine":(0.1,30),"Sodium":(100,180),
    "WBC":(0.1,200),"Hemoglobin":(1,25),"Lactate":(0.1,40),
}
# regex to VERIFY curated lab itemids against d_labitems (does not auto-override)
LAB_LABEL_REGEX = {
    "Bilirubin":r"bilirubin.*total|total.*bilirubin", "Albumin":r"^albumin",
    "INR":r"\binr\b", "PT":r"^pt$|prothrombin", "Platelets":r"platelet count|^platelets",
    "AST":r"aspartate aminotransferase|\bast\b", "ALT":r"alanine aminotransferase|\balt\b",
    "AlkPhos":r"alkaline phosphatase", "Ammonia":r"ammonia",
    "Creatinine":r"^creatinine", "Sodium":r"^sodium", "WBC":r"white blood cell|\bwbc\b",
    "Hemoglobin":r"^hemoglobin", "Lactate":r"^lactate|lactic acid",
}

# ============================================================
# CIRRHOSIS COHORT + COMPOSITE DECOMPENSATION (ICD prefixes, dots removed)
# ============================================================
CIRRHOSIS_ICD = {
    "icd9":  ("5712","5715","5716"),
    "icd10": ("K7030","K7031","K717","K740","K741","K742","K743","K744","K745","K746"),
}
DECOMP_ICD = {
    "ascites":        {"icd9":("78959","7895"),  "icd10":("R180","R188","R189")},
    "varices_bleed":  {"icd9":("4560","45620"),  "icd10":("I8501","I8511")},
    "hepatic_enceph": {"icd9":("5722",),         "icd10":("K720","K721","K729","K7290","K7291")},
    "sbp":            {"icd9":("56723",),        "icd10":("K652",)},
    "hepatorenal":    {"icd9":("5724",),         "icd10":("K767",)},
}
# Timestamped markers for LEAD-TIME ONLY (never model features)
DECOMP_DRUG_REGEX = {
    "hepatic_enceph":r"lactulose|rifaximin",
    "varices_bleed": r"octreotide|terlipressin|vasopressin",
    "sbp":           r"ceftriaxone|cefotaxime",   # low specificity — see design notes
}
PROC_LABEL_REGEX = {"paracentesis": r"paracentesis"}

def resolve_lab_itemids(verbose=True):
    """Verify curated LAB_ITEMIDS against hosp/d_labitems and report matches."""
    p = _csv(HOSP, "d_labitems")
    if not os.path.exists(p):
        print("  d_labitems not found — using curated itemids as-is.")
        return LAB_ITEMIDS
    d = _read(p)
    lab = "label" if "label" in d.columns else d.columns[-1]
    d[lab] = d[lab].astype(str)
    for feat, rgx in LAB_LABEL_REGEX.items():
        hits = d[d[lab].str.contains(rgx, case=False, regex=True, na=False)]
        ids  = hits["itemid"].tolist()
        curated = LAB_ITEMIDS.get(feat, [])
        ok = any(c in ids for c in curated)
        if verbose:
            print(f"  [{'OK' if ok else 'CHECK'}] {feat:<11} "
                  f"curated={curated} | dict matches={ids[:6]}")
    return LAB_ITEMIDS

print("Config loaded.")


MIMIC_ROOT = /Users/sumiyasayeed/Documents/mimic4/mimiciv
  hosp exists: True | icu exists: True
Config loaded.


## Data verification — run after config, before the pipeline

In [2]:
# ============================================================
# DATA VERIFICATION  — run AFTER the config cell, BEFORE the pipeline
# Confirms each table exists, has the expected columns/dtypes, and that
# the configured itemids/codes actually return data with plausible values.
# Big tables are sampled (SCAN_CHUNKS) so this stays fast.
# ============================================================
SCAN_CHUNKS  = 8     # CHUNK_SIZE chunks scanned in large tables (coverage vs speed)
SAMPLE_NROWS = 5

REQUIRED = {
    (HOSP,"diagnoses_icd"):   ["subject_id","hadm_id","icd_code","icd_version"],
    (HOSP,"d_icd_diagnoses"): ["icd_code","icd_version","long_title"],
    (HOSP,"patients"):        ["subject_id","gender","anchor_age","dod"],
    (HOSP,"admissions"):      ["subject_id","hadm_id","admittime","dischtime",
                               "hospital_expire_flag","race"],
    (HOSP,"labevents"):       ["subject_id","charttime","itemid","valuenum"],
    (HOSP,"d_labitems"):      ["itemid","label"],
    (HOSP,"prescriptions"):   ["subject_id","hadm_id","drug","starttime"],
    (ICU,"chartevents"):      ["subject_id","hadm_id","charttime","itemid","valuenum"],
    (ICU,"d_items"):          ["itemid","label"],
    (ICU,"procedureevents"):  ["subject_id","hadm_id","itemid","starttime"],
}

def _exists(dirpath, name):
    return any(os.path.exists(os.path.join(dirpath, name+ext)) for ext in (".csv",".csv.gz"))

# ---- 1) presence, columns, dtypes, sample ------------------
print("="*72); print("1) TABLE PRESENCE · COLUMNS · DTYPES · SAMPLE"); print("="*72)
status = {}
for (dirpath, name), need in REQUIRED.items():
    tag = f"{os.path.basename(dirpath)}/{name}"
    if not _exists(dirpath, name):
        print(f"  [MISSING] {tag}"); status[tag] = "MISSING"; continue
    head = _read(_csv(dirpath, name), nrows=SAMPLE_NROWS)
    cols = set(head.columns)
    miss = [c for c in need if c not in cols]
    flag = "OK" if not miss else "COLS-MISSING"
    status[tag] = flag
    print(f"  [{flag}] {tag}  ({len(head.columns)} columns)")
    if miss:
        print(f"        MISSING required columns: {miss}")
    have = [c for c in need if c in cols]
    print(f"        dtypes: {{{', '.join(f'{c}:{head[c].dtype}' for c in have)}}}")
    print(f"        sample: {head[have].head(2).to_dict('records')}")

# ---- helper: scan itemids + value ranges in a big table ----
def scan_itemids(dirpath, name, feature_map, value_ranges,
                 id_col="itemid", val_col="valuenum", n_chunks=SCAN_CHUNKS):
    want  = {i: f for f, ids in feature_map.items() for i in ids}
    found = {f: {"n": 0, "vmin": np.inf, "vmax": -np.inf, "vals": []} for f in feature_map}
    last  = 0
    for j, chunk in enumerate(pd.read_csv(_csv(dirpath, name),
            usecols=[id_col, val_col], chunksize=CHUNK_SIZE, low_memory=False)):
        last = j + 1
        sub = chunk[chunk[id_col].isin(want)].dropna(subset=[val_col])
        if not sub.empty:
            sub = sub.copy(); sub["feature"] = sub[id_col].map(want)
            for f, grp in sub.groupby("feature"):
                v = grp[val_col].astype(float); d = found[f]
                d["n"] += len(v); d["vmin"] = min(d["vmin"], v.min()); d["vmax"] = max(d["vmax"], v.max())
                if len(d["vals"]) < 5000:
                    d["vals"].extend(v.sample(min(len(v), 2000)).tolist())
        if last >= n_chunks: break
    print(f"  (scanned first {last} chunk(s))")
    rows = []
    for f, d in found.items():
        if d["n"] == 0:
            rows.append((f, "NO-DATA", 0, None, None, None, None)); continue
        vals = np.array(d["vals"]); lo, hi = value_ranges.get(f, (None, None))
        inr  = None if lo is None else round(float(((vals >= lo) & (vals <= hi)).mean()*100), 1)
        rows.append((f, "OK", d["n"], round(float(d["vmin"]), 2), round(float(d["vmax"]), 2),
                     round(float(np.median(vals)), 2), inr))
    return pd.DataFrame(rows, columns=["feature","status","n_scanned","min","max","median","%in_range"])

# ---- 2) lab itemids & values -------------------------------
print("\n"+"="*72); print("2) LAB ITEMIDS & VALUE RANGES  (hosp/labevents)"); print("="*72)
if status.get("hosp/labevents") == "OK":
    print(scan_itemids(HOSP, "labevents", LAB_ITEMIDS, LAB_RANGE).to_string(index=False))
else:
    print("  skipped (table not OK)")

# ---- 3) vital itemids & values -----------------------------
print("\n"+"="*72); print("3) VITAL ITEMIDS & VALUE RANGES  (icu/chartevents)"); print("="*72)
if status.get("icu/chartevents") == "OK":
    print(scan_itemids(ICU, "chartevents", CHART_ITEMIDS, VITAL_RANGE).to_string(index=False))
else:
    print("  skipped (table not OK)")

# ---- 4) ICD coverage + example titles ----------------------
print("\n"+"="*72); print("4) ICD CODE COVERAGE  (diagnoses_icd)"); print("="*72)
if status.get("hosp/diagnoses_icd") == "OK":
    dx = _read(_csv(HOSP, "diagnoses_icd"), usecols=["icd_code","icd_version"])
    dx["icd_code"] = dx["icd_code"].astype(str).str.replace(".","",regex=False).str.upper()
    is9, is10 = dx["icd_version"] == 9, dx["icd_version"] == 10
    def _cnt(c9, c10):
        return int(((is9  & dx["icd_code"].str.startswith(tuple(c9))) |
                    (is10 & dx["icd_code"].str.startswith(tuple(c10)))).sum())
    print(f"  Cirrhosis code rows: {_cnt(CIRRHOSIS_ICD['icd9'], CIRRHOSIS_ICD['icd10']):,}")
    for ev, codes in DECOMP_ICD.items():
        print(f"    {ev:<16}: {_cnt(codes['icd9'], codes['icd10']):,} rows")
    if status.get("hosp/d_icd_diagnoses") == "OK":
        di = _read(_csv(HOSP, "d_icd_diagnoses"))
        di["icd_code"] = di["icd_code"].astype(str).str.replace(".","",regex=False).str.upper()
        pref = tuple(CIRRHOSIS_ICD["icd9"]) + tuple(CIRRHOSIS_ICD["icd10"])
        samp = di[di["icd_code"].str.startswith(pref)][["icd_code","icd_version","long_title"]]
        print("  Example matched cirrhosis titles (verify these are truly cirrhosis):")
        print(samp.head(12).to_string(index=False))
else:
    print("  skipped (table not OK)")

# ---- 5) lab itemid dictionary check ------------------------
print("\n"+"="*72); print("5) LAB ITEMID DICTIONARY CHECK  (d_labitems)"); print("="*72)
if status.get("hosp/d_labitems") == "OK":
    resolve_lab_itemids(verbose=True)
else:
    print("  skipped (table not OK)")

# ---- 6) drug & procedure marker coverage -------------------
print("\n"+"="*72); print("6) DRUG & PROCEDURE MARKER COVERAGE"); print("="*72)
if status.get("hosp/prescriptions") == "OK":
    full_rx = "|".join(DECOMP_DRUG_REGEX.values()); hits = tot = last = 0
    for j, chunk in enumerate(pd.read_csv(_csv(HOSP, "prescriptions"),
            usecols=["drug"], chunksize=CHUNK_SIZE, low_memory=False)):
        last = j + 1
        d = chunk["drug"].astype(str).str.lower(); tot += len(d)
        hits += int(d.str.contains(full_rx, regex=True, na=False).sum())
        if last >= SCAN_CHUNKS: break
    print(f"  prescriptions scanned ~{tot:,} rows | decompensation-drug matches: {hits:,}")
if status.get("icu/d_items") == "OK":
    di = _read(_csv(ICU, "d_items")); lab = "label" if "label" in di.columns else di.columns[-1]
    para = di[di[lab].astype(str).str.contains(PROC_LABEL_REGEX["paracentesis"], case=False, na=False)]
    print(f"  paracentesis itemids in d_items: {para['itemid'].tolist()}")

# ---- summary -----------------------------------------------
print("\n"+"="*72); print("SUMMARY"); print("="*72)
for k, v in status.items():
    print(f"  {v:<13} {k}")
print("\nReview any MISSING / COLS-MISSING / NO-DATA / low %in_range before running the pipeline.")


1) TABLE PRESENCE · COLUMNS · DTYPES · SAMPLE
  [OK] hosp/diagnoses_icd  (5 columns)
        dtypes: {subject_id:int64, hadm_id:int64, icd_code:int64, icd_version:int64}
        sample: [{'subject_id': 10000032, 'hadm_id': 22595853, 'icd_code': 5723, 'icd_version': 9}, {'subject_id': 10000032, 'hadm_id': 22595853, 'icd_code': 78959, 'icd_version': 9}]
  [OK] hosp/d_icd_diagnoses  (3 columns)
        dtypes: {icd_code:int64, icd_version:int64, long_title:object}
        sample: [{'icd_code': 10, 'icd_version': 9, 'long_title': 'Cholera due to vibrio cholerae'}, {'icd_code': 11, 'icd_version': 9, 'long_title': 'Cholera due to vibrio cholerae el tor'}]
  [OK] hosp/patients  (6 columns)
        dtypes: {subject_id:int64, gender:object, anchor_age:int64, dod:object}
        sample: [{'subject_id': 10000032, 'gender': 'F', 'anchor_age': 52, 'dod': '2180-09-09'}, {'subject_id': 10000048, 'gender': 'F', 'anchor_age': 23, 'dod': nan}]
  [OK] hosp/admissions  (16 columns)
        dtypes: {subjec

## Step 0 — Cirrhosis cohort, composite decompensation label, demographics, event anchors

In [3]:
# ============================================================
# STEP 0 — CIRRHOSIS COHORT + COMPOSITE DECOMPENSATION LABEL
# ============================================================

def _icd_prefix_mask(series, prefixes):
    s = series.astype(str).str.replace(".", "", regex=False).str.upper()
    return s.str.startswith(tuple(p.upper() for p in prefixes))

def discover_cirrhosis_cohort():
    dx = _read(_csv(HOSP, "diagnoses_icd"),
               usecols=["subject_id","hadm_id","icd_code","icd_version"])
    dx["icd_code"] = dx["icd_code"].astype(str).str.replace(".","",regex=False).str.upper()
    is9, is10 = dx["icd_version"] == 9, dx["icd_version"] == 10

    cirr_mask = ((is9  & _icd_prefix_mask(dx["icd_code"], CIRRHOSIS_ICD["icd9"])) |
                 (is10 & _icd_prefix_mask(dx["icd_code"], CIRRHOSIS_ICD["icd10"])))
    cirr = dx[cirr_mask][["subject_id","hadm_id"]].drop_duplicates()
    print(f"✅ Cirrhosis diagnoses: {len(cirr):,} admissions | "
          f"{cirr['subject_id'].nunique():,} patients")

    frames = []
    for ev, codes in DECOMP_ICD.items():
        m = ((is9  & _icd_prefix_mask(dx["icd_code"], codes["icd9"])) |
             (is10 & _icd_prefix_mask(dx["icd_code"], codes["icd10"])))
        sub = dx[m][["subject_id","hadm_id"]].drop_duplicates()
        sub[ev] = 1
        frames.append(sub)
    decomp = frames[0]
    for f in frames[1:]:
        decomp = decomp.merge(f, on=["subject_id","hadm_id"], how="outer")
    ev_cols = list(DECOMP_ICD.keys())
    for c in ev_cols:
        if c not in decomp.columns: decomp[c] = 0
    decomp[ev_cols] = decomp[ev_cols].fillna(0).astype(int)
    decomp["decompensated"] = (decomp[ev_cols].sum(axis=1) > 0).astype(int)

    cohort = cirr.merge(decomp, on=["subject_id","hadm_id"], how="left")
    for c in ev_cols + ["decompensated"]:
        cohort[c] = cohort[c].fillna(0).astype(int)

    adm = _read(_csv(HOSP, "admissions"),
                usecols=["subject_id","hadm_id","admittime","dischtime",
                         "hospital_expire_flag","race"])
    cohort = cohort.merge(adm, on=["subject_id","hadm_id"], how="left")
    cohort["admittime"] = pd.to_datetime(cohort["admittime"], errors="coerce")
    # index = earliest cirrhosis admission per patient
    cohort = cohort.sort_values("admittime").drop_duplicates("subject_id", keep="first")
    if MAX_PATIENTS:
        cohort = cohort.sort_values("subject_id").head(MAX_PATIENTS)
    cohort["patient_id"] = "p" + cohort["subject_id"].astype(str)
    print(f"✅ Cohort (index admission/patient): {len(cohort):,}  | "
          f"decompensated: {cohort['decompensated'].sum():,} "
          f"({cohort['decompensated'].mean()*100:.1f}%)")
    print("   Component counts:\n" + cohort[ev_cols].sum().to_string())
    return cohort

def load_demographics(cohort):
    pts = _read(_csv(HOSP, "patients"), usecols=["subject_id","gender","anchor_age","dod"])
    pts["dod"] = pd.to_datetime(pts["dod"], errors="coerce")
    out = cohort.merge(pts, on="subject_id", how="left")
    out["is_deceased"] = out["dod"].notna().astype(int)
    print(f"✅ Demographics merged. Deceased: {out['is_deceased'].mean()*100:.1f}%")
    return out

def load_event_anchors(cohort):
    """Earliest timestamped decompensation marker per patient (lead-time only)."""
    hadm_set = set(cohort["hadm_id"].dropna().astype(int))
    subj_set = set(cohort["subject_id"])
    markers = []

    pres = _csv(HOSP, "prescriptions")
    if os.path.exists(pres):
        full_rx = "|".join(DECOMP_DRUG_REGEX.values())
        for chunk in pd.read_csv(pres,
                usecols=["subject_id","hadm_id","drug","starttime"],
                chunksize=CHUNK_SIZE, low_memory=False):
            chunk = chunk[chunk["hadm_id"].isin(hadm_set)]
            if chunk.empty: continue
            chunk["drug"] = chunk["drug"].astype(str).str.lower()
            hit = chunk[chunk["drug"].str.contains(full_rx, regex=True, na=False)]
            if not hit.empty:
                h = hit[["subject_id"]].copy()
                h["marker_time"] = pd.to_datetime(hit["starttime"], errors="coerce")
                markers.append(h)
    else:
        print("  ⚠ prescriptions not found — drug markers skipped.")

    proc, ditems = _csv(ICU, "procedureevents"), _csv(ICU, "d_items")
    para_ids = []
    if os.path.exists(proc) and os.path.exists(ditems):
        di = _read(ditems)
        lab = "label" if "label" in di.columns else di.columns[-1]
        para_ids = di[di[lab].astype(str).str.contains(
            PROC_LABEL_REGEX["paracentesis"], case=False, na=False)]["itemid"].tolist()
        if para_ids:
            pe = _read(proc, usecols=["subject_id","hadm_id","itemid","starttime"])
            pe = pe[pe["hadm_id"].isin(hadm_set) & pe["itemid"].isin(para_ids)]
            if not pe.empty:
                p = pe[["subject_id"]].copy()
                p["marker_time"] = pd.to_datetime(pe["starttime"], errors="coerce")
                markers.append(p)
    print(f"  paracentesis itemids resolved: {para_ids}")

    if not markers:
        print("  ⚠ No timestamped markers found.")
        return pd.DataFrame(columns=["subject_id","first_decomp_marker_time"])
    allm = pd.concat(markers, ignore_index=True).dropna(subset=["marker_time"])
    first = (allm.groupby("subject_id")["marker_time"].min().reset_index()
                 .rename(columns={"marker_time":"first_decomp_marker_time"}))
    print(f"✅ Timestamped decompensation markers: {len(first):,} patients")
    return first

print("Step 0 functions defined.")


Step 0 functions defined.


## Step 1 — Vital-sign extraction

In [4]:
# ============================================================
# STEP 1 — VITAL-SIGN EXTRACTION (chartevents)
# ============================================================

def load_chartevents(cohort):
    path = _csv(ICU, "chartevents")
    if not os.path.exists(path):
        print("  ⚠ chartevents not found — vitals skipped.")
        return pd.DataFrame()
    all_ids  = {i for ids in CHART_ITEMIDS.values() for i in ids}
    hadm_set = set(cohort["hadm_id"].dropna().astype(int))
    keep = []
    print("  Reading chartevents in chunks...")
    for j, chunk in enumerate(pd.read_csv(path,
            usecols=["subject_id","hadm_id","charttime","itemid","valuenum"],
            chunksize=CHUNK_SIZE, low_memory=False)):
        m = (chunk["hadm_id"].isin(hadm_set) & chunk["itemid"].isin(all_ids)
             & chunk["valuenum"].notna())
        f = chunk[m]
        if not f.empty: keep.append(f)
        if (j+1) % 20 == 0: print(f"    ...chunk {j+1}")
    if not keep: return pd.DataFrame()
    df = pd.concat(keep, ignore_index=True)
    df["charttime"] = pd.to_datetime(df["charttime"], errors="coerce")
    df = df.dropna(subset=["charttime"])
    print(f"  chartevents rows: {len(df):,}")
    return df

def pivot_vitals(ce):
    if ce.empty: return pd.DataFrame()
    i2f = {i:f for f,ids in CHART_ITEMIDS.items() for i in ids}
    ce = ce.copy(); ce["feature"] = ce["itemid"].map(i2f)
    ce = ce.dropna(subset=["feature"])
    mask = pd.Series(True, index=ce.index)
    for f,(lo,hi) in VITAL_RANGE.items():
        sel = ce["feature"] == f
        mask &= ~(sel & ((ce["valuenum"] < lo) | (ce["valuenum"] > hi)))
    ce = ce[mask]
    ce["window"] = ce["charttime"].dt.floor(WINDOW)
    agg = (ce.groupby(["subject_id","hadm_id","window","feature"])["valuenum"]
             .mean().unstack("feature").reset_index()
             .rename(columns={"window":"timestamp"}))
    hrv = (ce[ce["feature"]=="HR_bpm"]
             .groupby(["subject_id","hadm_id","window"])["valuenum"].std()
             .reset_index().rename(columns={"window":"timestamp","valuenum":"HRV_proxy"}))
    agg = agg.merge(hrv, on=["subject_id","hadm_id","timestamp"], how="left")
    agg["HRV_proxy"] = agg["HRV_proxy"].fillna(0.05)
    agg["patient_id"] = "p" + agg["subject_id"].astype(str)
    print(f"✅ Vitals pivoted: {agg.shape}")
    return agg

print("Step 1 functions defined.")


Step 1 functions defined.


## Steps 2-3 — Liver lab panel, unified timeline, MELD-Na

In [5]:
# ============================================================
# STEP 2-3 — LIVER LAB PANEL + UNIFIED TIMELINE + MELD-Na
# ============================================================

def load_labs(cohort):
    path = _csv(HOSP, "labevents")
    if not os.path.exists(path):
        print("  ⚠ labevents not found.")
        return pd.DataFrame()
    all_ids  = {i for ids in LAB_ITEMIDS.values() for i in ids}
    subj_set = set(cohort["subject_id"])
    keep = []
    print("  Reading labevents in chunks...")
    for j, chunk in enumerate(pd.read_csv(path,
            usecols=["subject_id","charttime","itemid","valuenum"],
            chunksize=CHUNK_SIZE, low_memory=False)):
        m = (chunk["subject_id"].isin(subj_set) & chunk["itemid"].isin(all_ids)
             & chunk["valuenum"].notna())
        f = chunk[m]
        if not f.empty: keep.append(f)
        if (j+1) % 20 == 0: print(f"    ...chunk {j+1}")
    if not keep: return pd.DataFrame()
    df = pd.concat(keep, ignore_index=True)
    df["charttime"] = pd.to_datetime(df["charttime"], errors="coerce")
    df = df.dropna(subset=["charttime"])
    i2f = {i:f for f,ids in LAB_ITEMIDS.items() for i in ids}
    df["feature"] = df["itemid"].map(i2f)
    mask = pd.Series(True, index=df.index)
    for f,(lo,hi) in LAB_RANGE.items():
        sel = df["feature"] == f
        mask &= ~(sel & ((df["valuenum"] < lo) | (df["valuenum"] > hi)))
    df = df[mask]
    df["window"] = df["charttime"].dt.floor(WINDOW)
    piv = (df.groupby(["subject_id","window","feature"])["valuenum"]
             .mean().unstack("feature").reset_index()
             .rename(columns={"window":"timestamp"}))
    piv["patient_id"] = "p" + piv["subject_id"].astype(str)
    print(f"✅ Labs pivoted: {piv.shape}")
    return piv

def build_timeline(vitals, labs, cohort):
    frames = []
    if not vitals.empty: frames.append(vitals[["patient_id","subject_id","timestamp"]])
    if not labs.empty:   frames.append(labs[["patient_id","subject_id","timestamp"]])
    if not frames:
        print("  ⚠ No vitals or labs — empty timeline.")
        return pd.DataFrame()
    tl = pd.concat(frames, ignore_index=True).drop_duplicates()
    if not vitals.empty:
        tl = tl.merge(vitals.drop(columns=["hadm_id"], errors="ignore"),
                      on=["patient_id","subject_id","timestamp"], how="left")
    if not labs.empty:
        tl = tl.merge(labs.drop(columns=["subject_id"], errors="ignore"),
                      on=["patient_id","timestamp"], how="left")
    tl = tl.sort_values(["patient_id","timestamp"])
    lab_cols = [c for c in LAB_ITEMIDS if c in tl.columns]
    tl[lab_cols] = tl.groupby("patient_id")[lab_cols].ffill()
    # constrain to index-admission window (+/- 1 day buffer)
    tl = tl.merge(cohort[["subject_id","admittime","dischtime"]], on="subject_id", how="left")
    tl["timestamp"] = pd.to_datetime(tl["timestamp"], errors="coerce")
    tl["admittime"] = pd.to_datetime(tl["admittime"], errors="coerce")
    tl["dischtime"] = pd.to_datetime(tl["dischtime"], errors="coerce")
    keep = (tl["admittime"].isna() |
            ((tl["timestamp"] >= tl["admittime"] - pd.Timedelta("1D")) &
             (tl["dischtime"].isna() | (tl["timestamp"] <= tl["dischtime"] + pd.Timedelta("1D")))))
    tl = tl[keep].copy()
    print(f"✅ Unified timeline: {tl.shape} across {tl['patient_id'].nunique():,} patients")
    return tl

def add_meld_na(df):
    """MELD = 3.78*ln(bili)+11.2*ln(INR)+9.57*ln(creat)+6.43;
       MELD-Na = MELD + 1.32*(137-Na) - 0.033*MELD*(137-Na)."""
    b, i, c = df.get("Bilirubin"), df.get("INR"), df.get("Creatinine")
    if any(x is None for x in (b, i, c)):
        df["MELD_Na"] = np.nan
        return df
    bili = b.clip(lower=1.0); inr = i.clip(lower=1.0); cr = c.clip(lower=1.0, upper=4.0)
    meld = (3.78*np.log(bili) + 11.2*np.log(inr) + 9.57*np.log(cr) + 6.43).clip(6, 40)
    na = df.get("Sodium")
    if na is not None:
        nac = na.clip(lower=125, upper=137)
        df["MELD_Na"] = (meld + 1.32*(137-nac) - 0.033*meld*(137-nac)).clip(6, 40)
    else:
        df["MELD_Na"] = meld
    return df

print("Steps 2-3 functions defined.")


Steps 2-3 functions defined.


## Steps 5-6 — Bayesian decompensation HMM + glass-box explainability

In [6]:
# ============================================================
# STEP 5-6 — BAYESIAN DECOMPENSATION HMM + GLASS-BOX (LLR)
# ============================================================
# Latent states: 0 = Compensated, 1 = Decompensated.
# Emissions: population-z-scored physiology. Population stats are estimated
# from the cohort itself (data-driven), not hardcoded. Decompensation
# DIRECTION per feature is clinically specified below.

HMM_FEATURES = ["Bilirubin","Albumin","INR","Platelets","Creatinine","Sodium",
                "Ammonia","WBC","AST","ALT","Lactate","MAP","HR_bpm","SpO2",
                "Resp_rate","GCS_total","HRV_proxy"]

DECOMP_Z = {  # sign/magnitude of z at the decompensated state
    "Bilirubin":+2.0,"Albumin":-2.0,"INR":+2.0,"Platelets":-2.0,"Creatinine":+2.0,
    "Sodium":-2.0,"Ammonia":+2.0,"WBC":+2.0,"AST":+1.5,"ALT":+1.5,"Lactate":+2.0,
    "MAP":-2.0,"HR_bpm":+1.5,"SpO2":-1.5,"Resp_rate":+1.5,"GCS_total":-2.0,"HRV_proxy":-2.0,
}
FEATURE_WEIGHTS = {
    "Bilirubin":1.6,"Albumin":1.4,"INR":1.5,"Platelets":1.2,"Creatinine":1.4,
    "Sodium":1.3,"Ammonia":1.5,"WBC":1.1,"AST":0.8,"ALT":0.8,"Lactate":1.5,
    "MAP":1.4,"HR_bpm":1.0,"SpO2":1.1,"Resp_rate":1.0,"GCS_total":1.5,"HRV_proxy":1.0,
}
FEATURE_LABELS = {
    "Bilirubin":"Total Bilirubin","Albumin":"Albumin","INR":"INR","Platelets":"Platelets",
    "Creatinine":"Creatinine","Sodium":"Sodium","Ammonia":"Ammonia","WBC":"WBC","AST":"AST",
    "ALT":"ALT","Lactate":"Lactate","MAP":"Mean Arterial Pressure","HR_bpm":"Heart Rate",
    "SpO2":"Oxygen Saturation","Resp_rate":"Respiratory Rate","GCS_total":"GCS",
    "HRV_proxy":"HRV (variability)",
}

def estimate_population_stats(tl):
    stats = {}
    for f in HMM_FEATURES:
        if f in tl.columns and tl[f].notna().sum() > 100:
            v  = tl[f].astype(float)
            mu = float(v.median())
            sd = float(v.std())
            if not np.isfinite(sd) or sd < 1e-6: sd = 1.0
            stats[f] = (mu, sd)
    print(f"✅ Population stats estimated for {len(stats)} features: {list(stats)}")
    return stats

def decompensation_hmm(df, stats):
    df = df.sort_values("timestamp").reset_index(drop=True)
    belief = np.array([0.85, 0.15])            # cirrhotic baseline prior
    base_T = np.array([[0.92, 0.08], [0.10, 0.90]])
    probs = []
    for _, row in df.iterrows():
        T = base_T.copy()
        if belief[1] > 0.4:                    # momentum once decompensating
            T[1, 1], T[1, 0] = 0.95, 0.05
        pred = belief @ T
        lls = llu = 0.0
        for f in HMM_FEATURES:
            if f not in stats: continue
            val = row.get(f, np.nan)
            if pd.isna(val): continue
            mu, sd = stats[f]
            z = (val - mu) / sd
            w, zu = FEATURE_WEIGHTS[f], DECOMP_Z[f]
            lls += w * norm.logpdf(z, 0.0, 1.0)
            llu += w * norm.logpdf(z, zu, 1.5)
        lps = np.log(pred[0] + 1e-300) + lls
        lpu = np.log(pred[1] + 1e-300) + llu
        m = max(lps, lpu)
        ps, pu = np.exp(lps - m), np.exp(lpu - m)
        belief = np.array([ps, pu]) / (ps + pu + 1e-9)
        probs.append(belief[1])
    df["decomp_probability"] = probs
    df["alert_triggered"] = df["decomp_probability"] > 0.5
    return df

def run_hmm_all(tl, stats):
    out = []
    pids = tl["patient_id"].unique()
    for k, pid in enumerate(pids):
        out.append(decompensation_hmm(tl[tl["patient_id"] == pid].copy(), stats))
        if (k+1) % 1000 == 0: print(f"  [{k+1}/{len(pids)}]")
    res = pd.concat(out, ignore_index=True)
    print(f"✅ HMM complete. Alert rate: {res['alert_triggered'].mean():.3f}")
    return res

def explain(df, stats):
    df = df.reset_index(drop=True)
    exps = []
    for i in range(len(df)):
        if i == 0:
            exps.append("Initial Baseline"); continue
        d = df.loc[i, "decomp_probability"] - df.loc[i-1, "decomp_probability"]
        best, bllr = None, -np.inf
        for f in HMM_FEATURES:
            if f not in stats: continue
            val = df.loc[i].get(f, np.nan)
            if pd.isna(val): continue
            mu, sd = stats[f]
            z, zu, w = (val - mu) / sd, DECOMP_Z[f], FEATURE_WEIGHTS[f]
            llr = abs(w * (-0.5*((z-zu)/1.5)**2) - w * (-0.5*z**2))
            if llr > bllr: bllr, best = llr, f
        lab = FEATURE_LABELS.get(best, best) if best else "Unknown"
        if d > 0.05:    exps.append(f"Decompensation risk up: {lab}")
        elif d < -0.05: exps.append(f"Risk down: stabilised by {lab}")
        else:           exps.append("Stable")
    df["clinical_explanation"] = exps
    return df

def explain_all(res, stats):
    out = [explain(g.copy(), stats) for _, g in res.groupby("patient_id")]
    f = pd.concat(out, ignore_index=True)
    print("✅ Explainability (LLR) complete.")
    return f

print("Steps 5-6 functions defined.")


Steps 5-6 functions defined.


## Run pipeline

In [7]:
# ============================================================
# RUN PIPELINE
# ============================================================

def run_pipeline():
    print("\n===== PEMDT-Cirrhosis Pipeline =====\n")
    resolve_lab_itemids(verbose=True)

    print("\n-- Step 0: Cohort + label + anchors --")
    cohort  = discover_cirrhosis_cohort()
    cohort  = load_demographics(cohort)
    anchors = load_event_anchors(cohort)
    cohort  = cohort.merge(anchors, on="subject_id", how="left")
    cohort["los_days"] = ((pd.to_datetime(cohort["dischtime"]) -
                           pd.to_datetime(cohort["admittime"]))
                          .dt.total_seconds() / 86400)
    cohort.to_csv(OUT / "cirrhosis_cohort.csv", index=False)

    print("\n-- Step 1: Vitals --")
    vitals = pivot_vitals(load_chartevents(cohort)); gc.collect()
    print("\n-- Step 2: Labs --")
    labs = load_labs(cohort)
    print("\n-- Step 3: Timeline + MELD-Na --")
    tl = build_timeline(vitals, labs, cohort); del vitals, labs; gc.collect()
    if tl.empty:
        print("⚠ Empty timeline — aborting."); return None, None, cohort
    tl = add_meld_na(tl)
    tl.to_csv(OUT / "cirrhosis_timeline.csv", index=False)

    print("\n-- Step 4: Population stats --")
    stats = estimate_population_stats(tl)
    (OUT / "population_stats.json").write_text(json.dumps(stats, indent=2))

    print("\n-- Step 5: Decompensation HMM --")
    res = run_hmm_all(tl, stats); del tl; gc.collect()
    print("\n-- Step 6: Explainability --")
    res = explain_all(res, stats)

    res = res.merge(
        cohort[["subject_id","decompensated","is_deceased","gender","anchor_age",
                "los_days","first_decomp_marker_time","admittime"] + list(DECOMP_ICD.keys())],
        on="subject_id", how="left")
    res.to_csv(OUT / "cirrhosis_dashboard.csv", index=False)

    has_meld = "MELD_Na" in res.columns
    agg = dict(
        mean_decomp=("decomp_probability","mean"),
        peak_decomp=("decomp_probability","max"),
        std_decomp =("decomp_probability","std"),
        alert_rate =("alert_triggered","mean"),
        n_windows  =("decomp_probability","count"),
        decompensated=("decompensated","first"),
        is_deceased=("is_deceased","first"),
        anchor_age =("anchor_age","first"),
        gender     =("gender","first"),
        los_days   =("los_days","first"),
    )
    if has_meld:
        agg["meld_na_max"]  = ("MELD_Na","max")
        agg["meld_na_mean"] = ("MELD_Na","mean")
    summ = res.groupby("patient_id").agg(**agg).reset_index()
    summ.to_csv(OUT / "cirrhosis_patient_summary.csv", index=False)

    print("\n✅ Pipeline complete. Outputs:")
    for f in ["cirrhosis_cohort.csv","cirrhosis_timeline.csv","cirrhosis_dashboard.csv",
              "cirrhosis_patient_summary.csv","population_stats.json"]:
        print("   ", f)
    return res, summ, cohort

if os.path.exists(HOSP):
    res, summ, cohort = run_pipeline()
else:
    print(f"MIMIC-IV not found at {MIMIC_ROOT}. Edit CANDIDATE_ROOTS in the config cell.")



===== PEMDT-Cirrhosis Pipeline =====

  [OK] Bilirubin   curated=[50885] | dict matches=[50838, 50885, 51028, 51049, 51783, 51812]
  [OK] Albumin     curated=[50862] | dict matches=[50835, 50862, 51019, 51025, 51046, 51069]
  [OK] INR         curated=[51237] | dict matches=[51237, 51675]
  [OK] PT          curated=[51274] | dict matches=[51274, 52921]
  [OK] Platelets   curated=[51265] | dict matches=[51265, 53189]
  [OK] AST         curated=[50878] | dict matches=[50878]
  [OK] ALT         curated=[50861] | dict matches=[50861, 53084]
  [OK] AlkPhos     curated=[50863] | dict matches=[50863, 51241, 53086]
  [OK] Ammonia     curated=[50866] | dict matches=[50866]
  [OK] Creatinine  curated=[50912] | dict matches=[50841, 50912, 51021, 51032, 51052, 51080]
  [OK] Sodium      curated=[50983] | dict matches=[50824, 50834, 50848, 50983, 51042, 51058]
  [OK] WBC         curated=[51301, 51300] | dict matches=[51300, 51301, 51516, 51517, 51518, 51755]
  [OK] Hemoglobin  curated=[51222] | dict

## Validation

In [8]:
# ============================================================
# VALIDATION — discrimination of decomp_probability vs composite label
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                             precision_recall_curve, brier_score_loss)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

VAL_OUT = Path("validation_output"); VAL_OUT.mkdir(exist_ok=True)

ps = pd.read_csv("cirrhosis_patient_summary.csv")
ps["std_decomp"] = ps["std_decomp"].fillna(0)
y = ps["decompensated"].astype(int).values
print(f"Patients: {len(ps):,} | decompensated: {y.sum():,} ({y.mean()*100:.1f}%)\n")

def boot_ci(yt, ys, n=1000, seed=42):
    rng = np.random.default_rng(seed); s = []
    for _ in range(n):
        idx = rng.integers(0, len(yt), len(yt))
        if not (1 < yt[idx].sum() < len(idx) - 1): continue
        s.append(roc_auc_score(yt[idx], ys[idx]))
    return (np.percentile(s, [2.5, 97.5]) if s else (np.nan, np.nan))

print("--- HMM discrimination (raw) ---")
for col in ["mean_decomp","peak_decomp"]:
    auc = roc_auc_score(y, ps[col]); lo, hi = boot_ci(y, ps[col].values)
    pr  = average_precision_score(y, ps[col])
    print(f"  {col:<12} AUC={auc:.3f} [{lo:.3f}-{hi:.3f}]  PR-AUC={pr:.3f}")

if "meld_na_max" in ps.columns and ps["meld_na_max"].notna().sum() > 50:
    m = ps.dropna(subset=["meld_na_max"])
    if m["decompensated"].nunique() > 1:
        print(f"\n--- MELD-Na clinical baseline ---")
        print(f"  meld_na_max  AUC={roc_auc_score(m['decompensated'], m['meld_na_max']):.3f}")

print("\n--- Calibrated combination (Platt) ---")
FEATS = ["mean_decomp","peak_decomp","alert_rate","anchor_age"]
d = ps.dropna(subset=FEATS + ["decompensated"])
Xs = StandardScaler().fit_transform(d[FEATS]); yy = d["decompensated"].astype(int).values
lr = LogisticRegression(max_iter=1000).fit(Xs, yy)
cal = lr.predict_proba(Xs)[:, 1]
print(f"  AUC={roc_auc_score(yy, cal):.3f}  Brier={brier_score_loss(yy, cal):.3f}")
for f, c in zip(FEATS, lr.coef_[0]):
    print(f"    {f:<14}{c:+.3f}")

fpr, tpr, _ = roc_curve(y, ps["mean_decomp"])
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"HMM mean_decomp AUC={roc_auc_score(y, ps['mean_decomp']):.3f}")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("PEMDT-Cirrhosis — Decompensation Discrimination"); plt.legend()
plt.tight_layout(); plt.savefig(VAL_OUT / "roc_decompensation.png", dpi=150); plt.close()

# Lead-time: first alert vs first timestamped decompensation marker
dash = pd.read_csv("cirrhosis_dashboard.csv",
        usecols=["patient_id","timestamp","alert_triggered","first_decomp_marker_time"],
        low_memory=False)
dash["timestamp"] = pd.to_datetime(dash["timestamp"], errors="coerce")
dash["first_decomp_marker_time"] = pd.to_datetime(dash["first_decomp_marker_time"], errors="coerce")
lead = []
for pid, g in dash[dash["first_decomp_marker_time"].notna()].groupby("patient_id"):
    g = g.sort_values("timestamp")
    a = g[g["alert_triggered"]]
    if a.empty: continue
    fa, mk = a["timestamp"].iloc[0], g["first_decomp_marker_time"].iloc[0]
    if pd.notna(fa) and pd.notna(mk) and fa <= mk:
        lead.append((mk - fa).total_seconds() / 3600)
if lead:
    lead = np.array(lead)
    print(f"\n--- Lead-time to decompensation marker (n={len(lead)}) ---")
    print(f"  median={np.median(lead):.1f} h  "
          f"IQR=[{np.percentile(lead,25):.1f}-{np.percentile(lead,75):.1f}] h")
else:
    print("\n(No overlapping alerts/markers for lead-time.)")

print("\n✅ Validation complete →", VAL_OUT)


Patients: 7,869 | decompensated: 2,981 (37.9%)

--- HMM discrimination (raw) ---
  mean_decomp  AUC=0.677 [0.664-0.688]  PR-AUC=0.543
  peak_decomp  AUC=0.690 [0.678-0.702]  PR-AUC=0.550

--- MELD-Na clinical baseline ---
  meld_na_max  AUC=0.750

--- Calibrated combination (Platt) ---
  AUC=0.595  Brier=0.223
    mean_decomp   +0.429
    peak_decomp   +0.419
    alert_rate    -0.361
    anchor_age    -0.038

--- Lead-time to decompensation marker (n=514) ---
  median=8.0 h  IQR=[3.2-20.8] h

✅ Validation complete → validation_output
